In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r'data/undefind_DSMED.csv', sep=';')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36297 entries, 0 to 36296
Data columns (total 13 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ID истории болезни                    36297 non-null  object 
 1   Осн. диаг. при выписке МКБ10 (текст)  36297 non-null  object 
 2   Заголовок документа                   36297 non-null  object 
 3   Кол. лаб. показатель                  36236 non-null  object 
 4   Значение кол. показателя              36236 non-null  float64
 5   Ед. изм. кол. показателя              36236 non-null  object 
 6   Норма кол. показателя                 36236 non-null  object 
 7   Флаг нормы кол. показателя            36236 non-null  object 
 8   Кач. лаб. показатель                  7005 non-null   object 
 9   Значение кач. показателя              7005 non-null   object 
 10  Норма кач. показателя                 7005 non-null   object 
 11  Пол            

In [5]:
df.nunique()

ID истории болезни                       243
Осн. диаг. при выписке МКБ10 (текст)      11
Заголовок документа                        1
Кол. лаб. показатель                     131
Значение кол. показателя                2608
Ед. изм. кол. показателя                  11
Норма кол. показателя                    123
Флаг нормы кол. показателя                 4
Кач. лаб. показатель                       7
Значение кач. показателя                 155
Норма кач. показателя                      5
Пол                                        2
Дата рождения пациента                   104
dtype: int64

In [44]:
df['Кол. лаб. показатель'].value_counts()

Кол. лаб. показатель
Гемоглобин (HGB)                                        1114
Средний объем эритроцита (MCV)                          1114
Гематокрит (HCT)                                        1114
Среднее содержание гемоглобина в эритроците (MCH)       1114
Средняя концентрация гемоглобина в эритроците (MCHC)    1114
                                                        ... 
Тромбокрит                                                 1
Эозинофилы, абсолютное количество                          1
Базофилы, абсолютное количество                            1
СОЭ по Панченкову                                          1
MXD#                                                       1
Name: count, Length: 131, dtype: int64

In [35]:
pasp_df = df[['ID истории болезни', 'Пол', 'Дата рождения пациента']].drop_duplicates().copy()
pasp_df

,ID истории болезни,Пол,Дата рождения пациента
0,2e1d0b3f-488a-11ed-ab5a-0050568844e6,Ж,1962-09-22 00:00:00
154,24612d4d-c466-11ec-ab54-0050568844e6,Ж,1939-02-06 00:00:00
430,b0a85bb2-3404-11ed-ab56-0050568844e6,Ж,1962-09-22 00:00:00
433,09564dae-ebb9-11ec-ab56-0050568844e6,Ж,1947-04-25 00:00:00
461,2efd978b-cd3d-11ed-8604-005056880ecb,М,1958-02-16 00:00:00
...,...,...,...
35717,69714728-b691-11ee-8606-005056880ecb,М,1956-11-19 00:00:00
35803,ffdf0ad3-bb5b-11ee-ab6f-0050568844e6,М,1953-11-21 00:00:00
35946,20f82885-b5cf-11ee-ab6f-0050568844e6,М,1948-06-02 00:00:00
36146,6166e156-c1af-11ed-8602-005056880ecb,М,1982-02-11 00:00:00


In [36]:
df_d = df.pivot_table(values='Значение кол. показателя', columns='Кол. лаб. показатель', index='ID истории болезни')
df_d = df_d.merge(pasp_df, on='ID истории болезни', how='inner')

In [37]:
def check_data_quality(data,
                       freq_treshold=0.95,
                       nunique_threshold=0.95,
                       null_threshold=0):
    """Проводит поиск дублей, пропусков и неинформативных признаков и выводит результат в консоль.
    
    Неинформативным считается признак с долей уникальных значений или повторов выше установленного порога.
    
    Parameters
    ----------
        data : DataFrame
            Датафрейм для анализа
        freq_treshold : float
            Порог масимальной частоты встречаемости признака
        nunique_treshhold : float
            Порог максимальной уникальности признака
        null_threshold : float
            Порог максимального отстутсвия признака
    """
    
    # Поиск дублей
    try:
        print(f'Число найденных дублей: {
            data.duplicated().value_counts().loc[True]
            }\n')
    except KeyError:
        print('Количество дублей: 0\n')

    # Поиск пропущенных значений
    cols_null_percent = data.isnull().mean()
    cols_with_null = cols_null_percent[cols_null_percent > null_threshold]\
        .sort_values(ascending=False)
    if cols_with_null.shape[0]: display(cols_with_null)
    print(f'Количество признаков с пустыми значениями: {cols_with_null.shape[0]}\n')

    # Поиск неинформативных признаков
    low_information_cols = []
    bad_feat_flag = False
    # цикл по всем столбцам
    for col in data.columns:
        #наибольшая относительная частота в признаке
        top_freq = data[col].value_counts(normalize=True).max()
        #доля уникальных значений от размера признака
        nunique_ratio = data[col].nunique() / data[col].count()
        # сравниваем наибольшую частоту с порогом
        if top_freq > freq_treshold:
            bad_feat_flag = True
            low_information_cols.append(col)
            print(f'{col}: {top_freq:.2%}% одинаковых значений')
        # сравниваем долю уникальных значений с порогом
        if nunique_ratio > nunique_threshold:
            bad_feat_flag = True
            low_information_cols.append(col)
            print(f'{col}: {nunique_ratio:.2%} уникальных значений')
    if not bad_feat_flag:
        print('Неинформативных признаков не найдено.')

In [34]:
print('Проверка тренировочной выборки:'+'\n'+'-' * 30)
check_data_quality(df_d, null_threshold=0.2)

Проверка тренировочной выборки:
------------------------------
Количество дублей: 0



MXD#                                                    0.995868
Моноциты, относительное количество                      0.995868
Моноциты, абсолютное количество                         0.995868
Базофилы, относительное количество                      0.995868
Базофилы, абсолютное количество                         0.995868
                                                          ...   
Гемоглобин (HGB)                                        0.388430
Средний объем эритроцита (MCV)                          0.388430
Среднее содержание гемоглобина в эритроците (MCH)       0.388430
Средняя концентрация гемоглобина в эритроците (MCHC)    0.388430
Базофилы                                                0.309917
Length: 125, dtype: float64

Количество признаков с пустыми значениями: 125

ID истории болезни: 100.00% уникальных значений
HCT: 96.88% уникальных значений
LY%: 96.77% уникальных значений
MCV: 97.92% уникальных значений
MO%: 100.00% уникальных значений
MPV: 100.00% уникальных значений
MXD: 100.00% уникальных значений
MXD# : 100.00%% одинаковых значений
MXD# : 100.00% уникальных значений
MXD%: 95.24% уникальных значений
NE%: 100.00% уникальных значений
PLT: 100.00% уникальных значений
RBC: 100.00% уникальных значений
RDW-CV : 95.83% уникальных значений
RDW-SD: 100.00% уникальных значений
RET%: 95.65% уникальных значений
WBC: 98.96% уникальных значений
Абсолютное количество базофилов (BA#): 100.00% уникальных значений
Абсолютное количество лимфоцитов (LY#): 100.00% уникальных значений
Абсолютное количество моноцитов (MO#): 100.00% уникальных значений
Абсолютное количество нейтрофилов (NE#): 100.00% уникальных значений
Абсолютное количество эозинофилов (EO#): 100.00% уникальных значений
Базофилы, абсолютное количест